In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from models.linear_probes import linear_probe

c:\Users\anast\miniconda3\envs\bat-naturelm\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\anast\miniconda3\envs\bat-naturelm\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
c:\Users\anast\miniconda3\envs\bat-naturelm\lib\site-p

In [3]:
from models.feature_generation import build_feature_bank, extract_encoder, extract_feature,pool_features

In [4]:
from preprocessing.dataset import PipistrelleDataset

In [5]:
from evaluation.metrics import compute_cv_stats,plot_model_comparison,label_confusion

In [6]:
encoder_e0 = extract_encoder('effnetb0')
encoder_p2 = extract_encoder('perch2')

In [7]:
encoder_NL = extract_encoder('NLM_BEATs')

In [ ]:
batdata = PipistrelleDataset(data_input=r"C:\Users\anast\Desktop\College\BA6\Bachelor Project\Source Code\bat-social-call-classifier\data\bat_metadata.csv",
                             root_dir=r"C:\Users\anast\Desktop\College\BA6\Bachelor Project\Source Code\bat-social-call-classifier\data\xenocanto-dataset",
                             encoder = "NLM_BEATs",)


In [ ]:
import numpy as np
audio1 = batdata[0]
print(np.array(audio1[0]).shape)

In [8]:
batdata = PipistrelleDataset(data_input=r"C:\Users\anast\Desktop\College\BA6\Bachelor Project\Source Code\bat-social-call-classifier\data\bat_metadata.csv",
                             root_dir=r"C:\Users\anast\Desktop\College\BA6\Bachelor Project\Source Code\bat-social-call-classifier\data\xenocanto-dataset",
                             encoder = 'effnetb0')

features,labels = build_feature_bank(batdata, encoder_e0, "effnetb0", device='cpu')

Dataset type: <class 'preprocessing.dataset.PipistrelleDataset'>


Extracting effnetb0: 100%|██████████| 284/284 [07:34<00:00,  1.60s/it]


In [ ]:
#build_feature_bank(batdata, encoder_e0, "effnetb0", device='cpu')
windows, labels = batdata[0]
        
# Extract features for this specific file
feats = extract_feature(windows, encoder_NL, 'NLM_BEATs')

In [ ]:
print(feats.shape)

In [ ]:
features = [feats]

pooled = pool_features(features, windows=True, method='mean', encoder='NLM_BEATs')
pooled2 = pool_features(pooled, windows=False, method='mean', encoder='NLM_BEATs')
print(pooled2.shape)

In [ ]:
path = r"C:\Users\anast\Desktop\College\BA6\Bachelor Project\Source Code\bat-social-call-classifier\models\features"
X_eff0 = np.load(path + "\\X_features2_not_normalized.npy")
X_NLM = np.load(path + "\\X_features_NLM.npy")
X_per2 = np.load(path + "\\perch_features.npz")['features']
y = np.load(path + "\\Y_labels2_not_normalized.npy")
label_names = ['Type A', 'Type B', 'Type C', 'Type D', 'Echo']

In [ ]:
y_true,y_pred = linear_probe(pool_features(X_eff0, windows=False, method='mean', encoder='effnetb0'),y,balance=True)

In [ ]:
print(y_true[0].size/5)
print(y_pred['SVM'][0].size/5)

In [ ]:
y_pred_svm = y_pred['SVM']
y_pred_rf = y_pred['Random Forest'] 
y_pred_mlp = y_pred['MLP']

In [ ]:
labels = ['Type A', 'Type B', 'Type C', 'Type D', 'Echo']
results_vault = {
    "Perch 2.0 SVM": compute_cv_stats(y_true, y_pred_svm, label_names=labels),
    "Perch 2.0 RF": compute_cv_stats(y_true, y_pred_rf, label_names=labels),
    "Perch 2.0 MLP": compute_cv_stats(y_true, y_pred_mlp, label_names=labels),
    }

# 2. Call the plot
plot_model_comparison(
    all_results=results_vault, 
    metrics_to_plot=['Macro-AUC', 'cmAP'], # Choose which metrics to show
    title="Pipistrelle Classification: Encoder Comparison"
)

In [ ]:
from evaluation.metrics import plot_comprehensive_results,generate_metrics_table

plot_comprehensive_results(results_vault, labels=label_names, title="Pipistrelle Classification: Model Comparison")

In [ ]:

generate_metrics_table(results_vault,labels)[1]